In [ ]:
from skimage import io
import matplotlib.pyplot as plt
import jax.numpy as jnp
from fft import *
import jax

jax.config.update("jax_enable_x64", True)
jax.devices()

In [ ]:
N = 256

NA = 0.5
lambda_0 = 655e-9  # m
eps = 1e-12
unit_l = 1.0 * lambda_0  # m

k_0_SI = 2 * jnp.pi / lambda_0  # m-1
k_0 = k_0_SI * unit_l

# Follows setup in incoherent CRLB / RL paper
raw_img = io.imread("cameraman.png")[:, :]
min_img = jnp.min(raw_img)
max_img = jnp.max(raw_img)
normalized_img = (raw_img - min_img) / (max_img - min_img)
print(jnp.max(raw_img), jnp.min(raw_img))

# Higher = smaller, harder to image
size_factor = 8

pixel_size = lambda_0 / (size_factor * NA * unit_l)  # unit_l
W = pixel_size * N
X, Y, R = create_axes(256, 256, N=256, pixel_size=pixel_size)
f_x, f_y = ft_image_coordinates(X, 256, 256, pixel_size=pixel_size)

x = jnp.sqrt(normalized_img) * jnp.exp(jnp.pi * 1j * R / W)
k_x = 2 * jnp.pi * f_x
k_y = 2 * jnp.pi * f_y
k = jnp.sqrt(k_x**2 + k_y**2)

print(X.shape)
extent = [-W / 2, W / 2, -W / 2, W / 2]

plot_unit_l = unit_l * 1e6  # um
r_extent = [
    -W / 2 * plot_unit_l,
    W / 2 * plot_unit_l,
    -W / 2 * plot_unit_l,
    W / 2 * plot_unit_l,
]
k_min = jnp.min(k_x)
k_max = jnp.max(k_x)

k_extent = [k_min, k_max, k_min, k_max]
plt.imshow(jnp.abs(x) ** 2, extent=extent)
plt.colorbar()
plt.show()
plt.imshow(jnp.angle(x), extent=extent)
plt.colorbar()
plt.show()
x.shape

In [ ]:
from imaging import make_circular_mask
from matplotlib.patches import Circle

fig, ax = plt.subplots()
F_pupil = make_circular_mask(k, k_0 * NA)
my_circle = Circle((0, 0), k_0 * NA, linestyle="--", color="red", fill=False)
xhat = fft2_shifted(x)
ax.add_patch(my_circle)
plt.imshow(jnp.log(jnp.abs(xhat) ** 2), extent=k_extent)
plt.colorbar()
# plt.xlim((-k_0, k_0))
# plt.ylim((-k_0, k_0))

In [ ]:
fig, ax = plt.subplots()
F_pupil = make_circular_mask(k, k_0 * NA)
yhat = xhat * F_pupil
my_circle = Circle((0, 0), k_0 * NA, linestyle="--", color="red", fill=False)
xhat = fft2_shifted(x)
ax.add_patch(my_circle)
plt.imshow(jnp.log(jnp.abs(yhat) ** 2), extent=k_extent)
plt.colorbar()

In [ ]:
y = ifft2_shifted(yhat)
mu = jnp.abs(y) ** 2
plt.imshow(mu, extent=r_extent)

# Fisher information from $\hat{y}_k$.
For the (Poisson) counts $\mu$, it holds that
$$\mu = |y|^2 = |U^\dagger \hat{y}|^2$$

As usual, the Fisher information for some parameter $\theta$ of $\mu = \mu (\theta)$ is
$$F(\theta) = (\dfrac{\partial \mu}{\partial \theta})^2 \frac{1}{\mu}$$

In [ ]:
# Consider real changes to $yhat$
def mu_from_yhat(yhat):
    y = ifft2_shifted(yhat)
    return jnp.abs(y) ** 2


v = jnp.zeros_like(yhat).at[N // 2, N // 2].set(1.0)
mu, dmu = jax.jvp(mu_from_yhat, (yhat,), (v,))
eps = 1e-12
FI = jnp.sum(1 / jnp.maximum(mu, eps) * dmu**2)
FI

In [ ]:
# Consider real and imaginary changes to $yhat$
v_re = v
v_im = 1j * v

mu, dmu_re = jax.jvp(mu_from_yhat, (yhat,), (v_re,))
_, dmu_im = jax.jvp(mu_from_yhat, (yhat,), (v_im,))

w = 1.0 / jnp.maximum(mu, eps)

F_rr = jnp.sum(w * dmu_re * dmu_re)
F_ii = jnp.sum(w * dmu_im * dmu_im)
F_ri = jnp.sum(w * dmu_re * dmu_im)

# mu's appear to be more sensitive to imaginary component of central yhat
F_local = jnp.array([[F_rr, F_ri], [F_ri, F_ii]])

# Information determinant value of real and imaginary component together.
D = jnp.linalg.det(F_local)
F_local, D

# Gerchberg-Saxton reconstruction of complex $\hat{y}_k$ from two amplitude measurements

In [ ]:
def gerchberg_saxton(amp_src, amp_tgt, random_phase=True, iterations=1000):
    print(jnp.sqrt(jnp.mean((amp_src) ** 2)))
    print(jnp.sqrt(jnp.mean((amp_tgt) ** 2)))
    if random_phase:
        key = jax.random.key(0)
        # Uniform random phase in [-pi, pi)
        key, subkey = jax.random.split(key)
        phi = jax.random.uniform(subkey, amp_src.shape, minval=-jnp.pi, maxval=jnp.pi)

        # If you want the corresponding complex unit-modulus field exp(i*phi):
        a = jnp.exp(1j * phi) * F_pupil
    else:
        a = ifft2_shifted(amp_tgt * F_pupil)
    for i in range(iterations):
        b = amp_src * jnp.exp(1j * jnp.angle(a))
        c = fft2_shifted(b)
        d = amp_tgt * jnp.exp(1j * jnp.angle(c)) * F_pupil
        a = ifft2_shifted(d)

        # Use amplitude difference between c and given amp_tgt as convergence signal
        err_L2 = jnp.sqrt(jnp.mean((jnp.abs(a) - amp_src) ** 2))
        if i % 100 == 0:
            print(err_L2)
    yhat_est = amp_src * jnp.exp(1j * jnp.angle(a))
    return yhat_est


# Find necessary phase assuming uniform amplitude of x
y_complex = gerchberg_saxton(jnp.abs(y), jnp.abs(yhat))

In [ ]:
# y = ifft2_shifted(xhat * F_pupil)
plt.imshow(jnp.abs(y_complex) ** 2, extent=r_extent)
plt.show()
plt.imshow(jnp.angle(y_complex), extent=r_extent)
plt.colorbar()